# Clase 12 — Proyecto integrador y puente hacia los transformers

**Curso:** NLP Clásico con Python · **Bloque 5:** Cierre
**Tiempo estimado:** 120 minutos (la más densa del curso: es la de cierre) · **Capítulo del manual:** `manual/12_proyecto_puente_transformers.md`

## Qué vas a ver en esta clase

- Una *pipeline* completa de clasificación de reseñas (TF-IDF + regresión logística), reutilizando los ajustes de las clases 5 y 6.
- Un modelo *transformer* preentrenado (`nlptown/bert-base-multilingual-uncased-sentiment`) aplicado en **zero-shot**, sin ajustar nada al corpus del curso.
- La comparación real entre ambos, en el mismo conjunto de prueba: exactitud, F1, tiempo de inferencia y coste computacional.
- Un ejemplo de error concreto de cada método, para comparar cuánto se puede explicar cada uno.
- El puente (sin enseñarlo) hacia la arquitectura de los transformers: atención, preentrenamiento y ajuste fino.

**Requisitos:** clasificación de textos con TF-IDF y regresión logística (clase 5); análisis de sentimiento sobre reseñas (clase 6). **Librerías:** `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `transformers`, `torch`. **Datos:** `datos/resenas_es.csv`, `datos/resenas_it.csv` y `datos/plan_b_transformer_resenas.json` (predicciones precalculadas, por si no hay conexión); en Colab, sube la carpeta `datos/`. **Todas las cifras son un *ejemplo ilustrativo***, sobre un corpus sintético de 300 reseñas por idioma. **Duración de la ejecución:** unos segundos si el modelo ya está en caché; si hay que descargarlo (≈669 MB), unos minutos la primera vez.

> **Nota:** esta clase es más densa que las anteriores porque cierra el curso con un proyecto integrador. Si prefieres repartirla en dos sesiones, un corte natural es después de la sección 3 (antes de medir tiempos y errores).

---
## Celda 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
asegurar("torch", "torch")
asegurar("transformers", "transformers")
print("Librerías listas.")

In [ ]:
import os
import time
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

IDIOMAS = ("es", "it")   # dato-es

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

MODELO_HF = "nlptown/bert-base-multilingual-uncased-sentiment"

def estrellas_a_etiqueta(etiqueta_estrellas):
    """Traduce la salida del modelo ("1 star".."5 stars") a positivo/negativo/neutro con la regla 1-2->negativo, 3->neutro, 4-5->positivo."""
    n = int(etiqueta_estrellas[0])
    if n <= 2:
        return "negativo"
    if n == 3:
        return "neutro"
    return "positivo"

def particion(idioma):
    """Carga las reseñas y devuelve la misma partición 70/30 estratificada (semilla 42) que usa todo el capítulo."""
    df = pd.read_csv(cargar_datos(f"resenas_{idioma}.csv"))
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    return df, ent, prueba

def pipeline_clasico():
    """La misma configuración de TF-IDF + regresión logística de las clases 5 y 6."""
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
                      ("clf", LogisticRegression(max_iter=1000))])

def cargar_plan_b():
    """Carga las predicciones precalculadas del transformer (plan B, sin conexión)."""
    with open(cargar_datos("plan_b_transformer_resenas.json"), encoding="utf-8") as f:
        return json.load(f)

def predecir_transformer(clasificador_hf, prueba, idioma, plan_b):
    """Si el modelo está disponible, predice de verdad y cronometra; si no, usa las predicciones y el tiempo precalculados del plan B, alineados por id."""
    ids = prueba["id"].astype(str).tolist()
    datos_idioma = plan_b["idiomas"][idioma]
    if clasificador_hf is not None:
        t0 = time.perf_counter()
        salidas = clasificador_hf(prueba["texto"].tolist(), truncation=True, max_length=128)
        tiempo = time.perf_counter() - t0
        estrellas = [s["label"] for s in salidas]
    else:
        estrellas_por_id = dict(zip([str(x) for x in datos_idioma["id"]], datos_idioma["estrellas"]))
        estrellas = [estrellas_por_id[i] for i in ids]
        tiempo = datos_idioma["t_inferencia_transformer_total_s"]
    etiquetas = [estrellas_a_etiqueta(e) for e in estrellas]
    return estrellas, etiquetas, tiempo

print("Funciones auxiliares cargadas.")

---
## 1. El clásico: TF-IDF + regresión logística

Reutilizamos la configuración de las clases 5 y 6 (`TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)` + `LogisticRegression`), entrenada solo con el 70 % de cada idioma y evaluada en el 30 % restante (semilla 42, la misma partición para todo el capítulo).

In [ ]:
RESULT_CLASICO = {}
for idioma in IDIOMAS:
    df, ent, prueba = particion(idioma)
    pipe = pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred = pipe.predict(prueba["texto"])
    acc = accuracy_score(prueba["etiqueta"], pred)
    f1m = f1_score(prueba["etiqueta"], pred, average="macro")
    vocab = pipe.named_steps["tfidf"].get_feature_names_out()
    RESULT_CLASICO[idioma] = dict(df=df, ent=ent, prueba=prueba, pipe=pipe, pred=list(pred), acc=acc, f1m=f1m, vocab=len(vocab))
    print(idioma, "| entrenamiento:", len(ent), "| prueba:", len(prueba),
          "| vocabulario:", len(vocab), "| exactitud:", round(acc, 3), "| F1 macro:", round(f1m, 3))
    assert len(prueba) == 90 and len(ent) == 210          # 300 reseñas, partición 70/30
    assert 1 / 3 < acc <= 1.0                               # mejor que el azar (3 clases equilibradas)

### ¿Qué observas?

Con solo 210 reseñas de entrenamiento (ruidosas: erratas, mayúsculas irregulares, alargamientos), el modelo lineal recupera bastante más de dos tercios de los casos en los dos idiomas, muy por encima del 33 % del azar. Es la misma cifra, del mismo orden, que la validación cruzada documentada en `datos/LEEME.md`.

---
## 2. El transformer preentrenado, en zero-shot

`nlptown/bert-base-multilingual-uncased-sentiment` (licencia MIT, ≈669 MB, 167 360 261 parámetros) predice 1-5 estrellas; las traducimos a las tres clases del curso con la regla **1-2→negativo, 3→neutro, 4-5→positivo** (justificada en el capítulo, sección 3.2). El modelo nunca ve una reseña del corpus del curso: es una evaluación **zero-shot**.

In [ ]:
assert estrellas_a_etiqueta("1 star") == "negativo" and estrellas_a_etiqueta("2 stars") == "negativo"
assert estrellas_a_etiqueta("3 stars") == "neutro"
assert estrellas_a_etiqueta("4 stars") == "positivo" and estrellas_a_etiqueta("5 stars") == "positivo"
print("Regla de mapeo verificada.")

La siguiente celda **intenta** descargar/cargar el modelo real; si no hay red o falla la descarga, sigue con el **plan B** (predicciones precalculadas en `datos/plan_b_transformer_resenas.json`), sin interrumpir el notebook.

In [ ]:
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()   # barra de progreso silenciada: solo cambia la presentación, no el resultado
except Exception:
    pass
try:
    from transformers.utils import logging as _log_hf
    _log_hf.disable_progress_bar()
except Exception:
    pass
try:
    from transformers import pipeline as construir_pipeline_hf
    clasificador_hf = construir_pipeline_hf("sentiment-analysis", model=MODELO_HF)
    TRANSFORMER_OK = True
    print("Modelo transformer cargado desde la caché de Hugging Face.")
except Exception as error:
    clasificador_hf = None
    TRANSFORMER_OK = False
    print("No se pudo cargar el modelo (sin red o descarga fallida):", type(error).__name__)
    print("Se usará el PLAN B: predicciones precalculadas en datos/plan_b_transformer_resenas.json.")

PLAN_B = cargar_plan_b()
print("transformer en vivo:", TRANSFORMER_OK)

In [ ]:
RESULT_TRAFO = {}
for idioma in IDIOMAS:
    prueba = RESULT_CLASICO[idioma]["prueba"]
    estrellas, pred, tiempo = predecir_transformer(clasificador_hf, prueba, idioma, PLAN_B)
    acc = accuracy_score(prueba["etiqueta"], pred)
    f1m = f1_score(prueba["etiqueta"], pred, average="macro")
    RESULT_TRAFO[idioma] = dict(estrellas=estrellas, pred=pred, acc=acc, f1m=f1m, tiempo=tiempo)
    print(idioma, "| exactitud:", round(acc, 3), "| F1 macro:", round(f1m, 3),
          "| tiempo total (referencia CPU):", PLAN_B["idiomas"][idioma]["t_inferencia_transformer_total_s"], "s para", len(prueba), "reseñas")
    assert len(pred) == len(prueba) == 90
    assert 1 / 3 < acc <= 1.0

### ¿Qué observas?

Sin ver una sola reseña del corpus del curso, el *transformer* preentrenado obtiene una exactitud más alta que el clásico entrenado específicamente sobre él, en los dos idiomas (sección 3 del capítulo explica por qué: el dominio del modelo —reseñas de productos— coincide con el del corpus, y el clásico solo tuvo 210 ejemplos ruidosos para aprender).

---
## 3. La comparación, cara a cara

Una tabla y un gráfico con las cuatro cifras (2 métodos × 2 idiomas) sobre el mismo conjunto de prueba.

In [ ]:
filas = []
for idioma in IDIOMAS:
    filas.append(dict(idioma=idioma, metodo="clasico", exactitud=RESULT_CLASICO[idioma]["acc"], f1_macro=RESULT_CLASICO[idioma]["f1m"]))
    filas.append(dict(idioma=idioma, metodo="transformer", exactitud=RESULT_TRAFO[idioma]["acc"], f1_macro=RESULT_TRAFO[idioma]["f1m"]))
COMPARACION = pd.DataFrame(filas)
print(COMPARACION.round(3).to_string(index=False))
assert (COMPARACION.groupby("idioma")["exactitud"].apply(lambda s: s.iloc[1] > s.iloc[0])).all()   # en este corpus, el transformer gana en los dos idiomas

In [ ]:
fig, eje = plt.subplots(figsize=(7, 4))
ancho = 0.35
posiciones = np.arange(len(IDIOMAS))
for i, metodo in enumerate(("clasico", "transformer")):
    valores = [COMPARACION[(COMPARACION.idioma == idioma) & (COMPARACION.metodo == metodo)]["exactitud"].iloc[0] for idioma in IDIOMAS]
    eje.bar(posiciones + i * ancho, valores, ancho, label=metodo)
eje.set_xticks(posiciones + ancho / 2)
eje.set_xticklabels(IDIOMAS)
titulo = "Exactitud por idioma y método"
eje.set_title(titulo)
eje.set_ylim(0, 1.0)
eje.legend()
plt.tight_layout(); plt.show()

### ¿Qué observas?

El *transformer* está por delante en las dos barras de cada idioma. La diferencia (≈0,13 en español, ≈0,06 en italiano) es real en este experimento, pero recuerda la advertencia del capítulo: es el resultado de **este** corpus pequeño y de **este** dominio (reseñas de producto), no una ley general.

---
## 4. Tiempo de inferencia y coste computacional (CPU, referencia)

Medimos de verdad el tiempo de entrenamiento/predicción del clásico y el de predicción del *transformer*, y comprobamos que el clásico es más rápido; para que el notebook dé el mismo resultado impreso en cualquier ejecución (los milisegundos exactos varían de una máquina o de una ejecución a otra), mostramos las **cifras de referencia** registradas por el autor en CPU, ya validadas por esa misma comprobación en vivo.

In [ ]:
for idioma in IDIOMAS:
    ent, prueba, pipe = RESULT_CLASICO[idioma]["ent"], RESULT_CLASICO[idioma]["prueba"], RESULT_CLASICO[idioma]["pipe"]
    # Medimos de verdad (para comprobar el orden de magnitud); imprimimos las cifras de referencia del autor (CPU), estables entre ejecuciones
    t0 = time.perf_counter(); pipeline_clasico().fit(ent["texto"], ent["etiqueta"]); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); pipe.predict(prueba["texto"]); t_pred = time.perf_counter() - t0
    tiempo_trafo = RESULT_TRAFO[idioma]["tiempo"]
    assert t_pred < tiempo_trafo   # el clásico predice mucho más rápido en CPU (comprobado en vivo)
    ref = PLAN_B["idiomas"][idioma]
    print(idioma, "| clásico (referencia CPU): entrenar", ref["t_entrenar_clasico_ref_s"], "s, predecir", ref["t_predecir_clasico_ref_s"],
          f"s ({len(prueba)} reseñas) | transformer (referencia CPU): predecir", ref["t_inferencia_transformer_total_s"], "s")

IDIOMA_EJ = IDIOMAS[0]   # dato-es
n_params_trafo = PLAN_B["n_parametros"]
mb_trafo = PLAN_B["tamano_mb_fp32"]
n_params_clasico = 3 * RESULT_CLASICO[IDIOMA_EJ]["vocab"]
print("parámetros del transformer:", n_params_trafo, f"(~{mb_trafo} MB, fp32) | licencia:", PLAN_B["licencia"])
print("coeficientes del clásico, idioma de ejemplo:", IDIOMA_EJ, "|", n_params_clasico, "| razón:", round(n_params_trafo / n_params_clasico), "x")

### ¿Qué observas?

El clásico predice en milisegundos (o menos) sobre las 90 reseñas de prueba; el *transformer* tarda del orden de un segundo para el mismo lote. Y el *transformer* tiene decenas de miles de veces más parámetros que la regresión logística de este corpus. Ninguna cifra es "mala": el coste solo importa según el volumen y los recursos disponibles (capítulo, sección 7).

---
## 5. Facilidad de explicar los errores

**Clásico:** los coeficientes de la regresión logística se pueden leer palabra por palabra.

In [ ]:
pipe_ej = RESULT_CLASICO[IDIOMA_EJ]["pipe"]
vocab_ej = np.array(pipe_ej.named_steps["tfidf"].get_feature_names_out())
clf_ej = pipe_ej.named_steps["clf"]
for i, clase in enumerate(clf_ej.classes_):
    idx = np.argsort(-clf_ej.coef_[i])[:4]
    print(f"  {clase:>10}:", list(zip(vocab_ej[idx], np.round(clf_ej.coef_[i][idx], 2))))

In [ ]:
# Un error concreto del clásico: primera reseña de prueba mal clasificada, en IDIOMA_EJ
prueba_ej = RESULT_CLASICO[IDIOMA_EJ]["prueba"].reset_index(drop=True)
pred_clasico_ej = RESULT_CLASICO[IDIOMA_EJ]["pred"]
errores_clasico = [i for i in range(len(prueba_ej)) if pred_clasico_ej[i] != prueba_ej["etiqueta"].iloc[i]]
i = errores_clasico[0]
print("texto:", prueba_ej["texto"].iloc[i])
print("real:", prueba_ej["etiqueta"].iloc[i], "| predicho:", pred_clasico_ej[i])
print("explicación: se puede sumar el coeficiente de cada palabra del texto y comprobar el resultado (sección 6.1 del capítulo)")
assert len(errores_clasico) > 0   # el clásico no acierta el 100%

**Transformer:** no hay coeficientes por palabra; buscamos igualmente un error concreto en el conjunto de prueba, para verlo de cerca.

In [ ]:
pred_trafo_ej = RESULT_TRAFO[IDIOMA_EJ]["pred"]
estrellas_ej = RESULT_TRAFO[IDIOMA_EJ]["estrellas"]
errores_trafo = [i for i in range(len(prueba_ej)) if pred_trafo_ej[i] != prueba_ej["etiqueta"].iloc[i]]
j = errores_trafo[0]
print("texto:", prueba_ej["texto"].iloc[j])
print("real:", prueba_ej["etiqueta"].iloc[j], "| predicho:", pred_trafo_ej[j], f"({estrellas_ej[j]})")
print("explicación: no hay una tabla de pesos que lo justifique; solo la salida final del modelo (sección 6.2 del capítulo)")
assert len(errores_trafo) > 0

### ¿Qué observas?

El error del clásico se explica con números que puedes sumar a mano (los coeficientes de la sección anterior). El error del *transformer* solo se puede describir ("predijo mal"), no explicar con la misma precisión: no hay una tabla equivalente sin herramientas adicionales fuera del alcance de este curso.

---
## 6. Cuándo conviene cada enfoque

| Situación | Conviene |
|---|---|
| Pocos datos, dominio raro, hay que explicar cada decisión | **Clásico** |
| Recursos limitados (memoria, sin conexión, presupuesto de cómputo) | **Clásico** |
| Ningún dato propio, dominio parecido al de un modelo ya entrenado | **Transformer preentrenado** |
| Vocabulario real muy variado (sinónimos, paráfrasis, errores) | **Transformer** |
| Datos etiquetados suficientes y tiempo para ajustar el modelo | **Ajustar (*fine-tuning*) un transformer**, fuera de este curso |

## 7. Puente hacia los transformers

Todo lo visto en el curso representa el texto con un vector por palabra o par de palabras, sin que el contexto lo cambie. Los transformers añaden **atención**: cada palabra ajusta su representación según las palabras que la acompañan en cada frase concreta. Se construyen en dos fases: **preentrenamiento** (aprender de texto sin etiquetar, a gran escala) y **ajuste fino** (*fine-tuning*, especializarse en una tarea con relativamente pocos ejemplos etiquetados). Ninguna de las dos se enseña en este curso: es la puerta de entrada al siguiente.

---
## Resumen

- Una *pipeline* de NLP tiene decisiones en cada paso (preprocesamiento, representación, clasificación, evaluación, análisis de errores) que hay que justificar, no dar por sentadas.
- Un modelo *transformer* preentrenado se usa con `pipeline(...)` de Hugging Face en unas pocas líneas, sin entrenar nada.
- En este corpus (reseñas sintéticas, dominio de productos), el *transformer* en zero-shot **superó** al clásico entrenado específicamente: un resultado real, no forzado, con una explicación identificable (capítulo, sección 4).
- El clásico se explica con coeficientes; el *transformer*, no, sin herramientas adicionales: es una diferencia práctica a la hora de elegir.
- Los transformers añaden atención sobre las representaciones que ya conoces; se construyen con preentrenamiento y ajuste fino, el tema del curso siguiente.

**Esto cierra el curso.** El notebook de práctica (`clase12_practica_proyecto_puente_transformers.ipynb`) repite el proyecto sobre un corte distinto de los datos y con más detalle, con ejercicios paso a paso.